# Amazon Bedrock AgentCore Runtime의 Strands Agents Streaming 응답

## 개요

이 튜토리얼에서는 Amazon Bedrock AgentCore Runtime을 사용하여 streaming 응답을 구현하는 방법을 알아봅니다. 이 예제는 부분 결과가 준비되는 즉시 stream하여 대량의 콘텐츠를 생성하거나 처리 시간이 오래 걸리는 작업에서 더 빠르게 반응하는 사용자 경험을 제공하는 방법을 보여줍니다.


### 튜토리얼 세부 정보

|항목| 세부 정보|
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | Streaming 기반 대화형|
| 에이전트 유형       | 단일           |
| Agentic Framework   | Strands Agents |
| LLM 모델            | Anthropic Claude Haiku 4.5 |
| 튜토리얼 구성 요소  | AgentCore Runtime, Strands Agent, Amazon Bedrock 모델을 사용한 Streaming 응답 |
| 튜토리얼 분야       | 산업 공통                                                                        |
| 예제 난이도         | 쉬움                                                                             |
| 사용 SDK            | Amazon BedrockAgentCore Python SDK 및 boto3|

### 튜토리얼 아키텍처

이 튜토리얼에서는 streaming agent를 AgentCore Runtime에 배포하는 방법을 설명합니다. 

데모에서는 streaming 기능이 있는 Amazon Bedrock 모델 기반 Strands Agent를 사용합니다.

예제에서는 `get_weather`와 `get_time` 두 가지 tool 및 streaming 응답 기능이 있는 간단한 에이전트를 사용합니다.

    
<div style="text-align:left">
    <img src="images/architecture_runtime.png" width="60%"/>
</div>

### 튜토리얼 주요 기능

* Amazon Bedrock AgentCore Runtime 에이전트의 Streaming 응답
* 실시간 부분 결과 전달
* streaming과 Amazon Bedrock 모델 사용
* async streaming을 지원하는 Strands Agents 사용

## 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Python 3.10+
* AWS credentials
* Amazon Bedrock AgentCore SDK
* Strands Agents
* 실행 중인 Docker

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## AgentCore Runtime 배포를 위한 streaming agent 준비

이제 streaming agent를 AgentCore Runtime에 배포합니다. entrypoint 함수에서 async generator 또는 yield statement를 사용하면 AgentCore SDK가 streaming 기능을 자동으로 처리합니다.

streaming 구현의 핵심 사항은 다음과 같습니다.
* entrypoint 함수에 `async def` 사용
* chunk가 준비되는 즉시 stream하도록 `yield` 사용
* AgentCore SDK가 Server-Sent Events(SSE) 형식을 자동 처리
* client가 Content-Type: text/event-stream 응답 수신

### Amazon Bedrock 모델 및 Streaming 기반 Strands Agents
Amazon Bedrock 모델을 사용하는 Strands Agent의 streaming 구현을 살펴봅니다.

In [ ]:
%%writefile strands_claude_streaming.py
from strands import Agent, tool
from strands_tools import calculator # calculator tool 가져오기
import argparse
import json
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands.models import BedrockModel
import asyncio
from datetime import datetime

app = BedrockAgentCoreApp()

# custom tool 생성 
@tool
def weather():
    """ Get weather """ # Dummy 구현
    return "sunny"

@tool
def get_time():
    """ Get current time """
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

model_id = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
model = BedrockModel(
    model_id=model_id,
)
agent = Agent(
    model=model,
    tools=[
        calculator, weather, get_time
    ],
    system_prompt="""You're a helpful assistant. You can do simple math calculations, 
    tell the weather, and provide the current time."""
)

@app.entrypoint
async def strands_agent_bedrock_streaming(payload):
    """
    스트리밍 기능으로 에이전트를 호출합니다.
    이 함수는 비동기 제너레이터를 사용해 AgentCore Runtime에서
    스트리밍 응답을 구현하는 방법을 보여 줍니다.
    """
    user_input = payload.get("prompt")
    print("User input:", user_input)
    
    try:
        # 각 chunk가 준비되는 즉시 stream
        async for event in agent.stream_async(user_input):
            if "data" in event:
                yield event["data"]
            
    except Exception as e:
        # streaming context에서 오류를 적절히 처리
        error_response = {"error": str(e), "type": "stream_error"}
        print(f"Streaming error: {error_response}")
        yield error_response

if __name__ == "__main__":
    app.run()

## AgentCore Runtime의 Streaming 이해

AgentCore Runtime에서 streaming을 사용하면 다음 작업이 자동으로 수행됩니다.

### Server-Sent Events(SSE) 형식
* AgentCore SDK가 yield된 데이터를 SSE 형식으로 자동 변환
* 각 yield가 SSE stream의 `data: ` event로 변환
* Content-Type이 `text/event-stream`으로 자동 설정

### Client 처리
* 에이전트가 요청을 처리하는 동안 client가 실시간 update 수신
* 점진적 응답 표시와 더 나은 사용자 경험 제공
* 전체 응답이 준비되기 전에 client가 부분 결과 처리 가능

### 오류 처리
* Streaming 응답에 적절한 오류 처리 포함
* 오류를 stream의 일부로 yield 가능
* 함수가 완료되거나 처리되지 않은 exception이 발생하면 stream 종료

## AgentCore Runtime에 streaming agent 배포

`CreateAgentRuntime` operation은 container image, 환경 변수, 암호화 설정을 지정할 수 있는 포괄적인 구성 옵션을 지원합니다. protocol 설정(HTTP, MCP)과 권한 부여 메커니즘을 구성하여 client가 에이전트와 통신하는 방식도 제어할 수 있습니다. 

**참고:** 운영 환경에서는 코드를 container로 package하고 CI/CD pipeline과 IaC를 사용하여 ECR에 push하는 것이 좋습니다.

이 튜토리얼에서는 Amazon Bedrock AgentCore Python SDK를 사용하여 artifact를 간편하게 package하고 AgentCore Runtime에 배포합니다.

### AgentCore Runtime 배포 구성

다음으로 starter toolkit을 사용하여 entrypoint, 앞에서 생성한 execution role, requirements 파일로 AgentCore Runtime 배포를 구성합니다. 실행 시 Amazon ECR repository를 자동으로 생성하도록 starter toolkit도 구성합니다.

configure 단계에서는 애플리케이션 코드를 기반으로 Dockerfile이 생성됩니다.

<div style="text-align:left">
    <img src="images/configure.png" width="60%"/>
</div>

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
region

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="strands_claude_streaming.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="strands_claude_streaming",
)
response

### AgentCore Runtime에 streaming agent 실행

Dockerfile이 준비되었으므로 AgentCore Runtime에 streaming agent를 실행합니다. 이 과정에서 Amazon ECR repository와 AgentCore Runtime이 생성됩니다.

<div style="text-align:left">
    <img src="images/launch.png" width="85%"/>
</div>

In [ ]:
launch_result = agentcore_runtime.launch()

### AgentCore Runtime 상태 확인
AgentCore Runtime을 배포했으므로 배포 상태를 확인합니다.

In [ ]:
import time

status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]
end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(status)
status

### Streaming으로 AgentCore Runtime 호출

이제 payload로 AgentCore Runtime을 호출하고 streaming 응답을 받을 수 있습니다.

<div style="text-align:left">
    <img src="images/invoke.png" width="85%"/>
</div>

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "what the weather is like?"})
invoke_response

### boto3로 AgentCore Runtime Streaming 호출

AgentCore Runtime이 생성되었으므로 어떤 AWS SDK로도 호출할 수 있습니다. streaming 응답을 사용하려면 Server-Sent Events 형식을 처리해야 합니다.

In [ ]:
import boto3
import json
from IPython.display import Markdown, display

agent_arn = launch_result.agent_arn
agentcore_client = boto3.client("bedrock-agentcore", region_name=region)

# streaming 응답을 위해 EventStream 처리
boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "How much is 2+1"}),
)

# 응답이 streaming인지 확인
if "text/event-stream" in boto3_response.get("contentType", ""):
    print("Processing streaming response with boto3:")
    content = []
    for line in boto3_response["response"].iter_lines(chunk_size=1):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                data = line[6:].replace('"', "")  # "data: " prefix 제거
                print(f"Received streaming chunk: {data}")
                content.append(data.replace('"', ""))

    # 전체 streamed 응답 표시
    full_response = " ".join(content)
    display(Markdown(full_response))
else:
    # non-streaming 응답 처리
    try:
        events = []
        for event in boto3_response.get("response", []):
            events.append(event)
    except Exception as e:
        events = [f"Error reading EventStream: {e}"]

    if events:
        try:
            response_data = json.loads(events[0].decode("utf-8"))
            display(Markdown(response_data))
        except:
            print(f"Raw response: {events[0]}")

## Streaming 응답의 이점

Streaming 응답은 다음과 같은 주요 이점을 제공합니다.

### 사용자 경험
* **즉각적인 feedback**: 부분 결과가 준비되는 즉시 사용자에게 표시
* **체감 성능**: 총 소요 시간이 같아도 응답이 더 빠르게 느껴짐
* **점진적 표시**: 긴 응답을 단계적으로 표시 가능

### 기술적 이점
* **효율적인 Memory 사용**: 전체를 memory에 올리지 않고 대용량 응답 처리
* **Timeout 방지**: 장기 실행 작업의 timeout 방지
* **실시간 처리**: 실시간 데이터가 준비되는 즉시 처리

### 사용 사례
* **콘텐츠 생성**: 장문 작성, 보고서, 문서
* **데이터 분석**: 복잡한 계산의 점진적 결과
* **Multi-step Workflow**: 복잡한 agent reasoning 진행 상황 표시
* **실시간 Monitoring**: monitoring agent의 실시간 update

## 리소스 정리(선택 사항)

이제 생성한 AgentCore Runtime을 정리합니다.

In [ ]:
launch_result.ecr_uri, launch_result.agent_id, launch_result.ecr_uri.split("/")[1]

In [ ]:
agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)
ecr_client = boto3.client("ecr", region_name=region)

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,
)

response = ecr_client.delete_repository(repositoryName=launch_result.ecr_uri.split("/")[1], force=True)

# 축하합니다!

Amazon Bedrock AgentCore Runtime을 사용하여 streaming agent를 성공적으로 구현하고 배포했습니다. 

## 학습한 내용:
* async generator를 사용하여 streaming 응답을 구현하는 방법
* AgentCore Runtime이 SSE 형식을 자동으로 처리하는 방법
* client 측에서 streaming 응답을 처리하는 방법
* 사용자 경험 및 성능 측면에서 streaming의 이점

## 다음 단계:
* 사용 사례에 맞는 다양한 streaming pattern 실험
* 복잡한 multi-step workflow용 custom streaming logic 구현
* streaming을 Memory 및 Gateway 같은 다른 AgentCore 기능과 결합하는 방법 탐색
* 더 나은 UX를 위한 client-side streaming 시각화 구현 검토